# 三数之和

给你一个整数数组 nums ，判断是否存在三元组 [nums[i], nums[j], nums[k]] 满足 i != j、i != k 且 j != k ，同时还满足 nums[i] + nums[j] + nums[k] == 0 。请你返回所有和为 0 且不重复的三元组。

注意：答案中不可以包含重复的三元组。


示例 1：

输入：nums = [-1,0,1,2,-1,-4]

输出：[[-1,-1,2],[-1,0,1]]

解释：

nums[0] + nums[1] + nums[2] = (-1) + 0 + 1 = 0 。

nums[1] + nums[2] + nums[4] = 0 + 1 + (-1) = 0 。

nums[0] + nums[3] + nums[4] = (-1) + 2 + (-1) = 0 。

不同的三元组是 [-1,0,1] 和 [-1,-1,2] 。

注意，输出的顺序和三元组的顺序并不重要。


示例 2：

输入：nums = [0,1,1]

输出：[]

解释：唯一可能的三元组和不为 0 。

示例 3：

输入：nums = [0,0,0]

输出：[[0,0,0]]

解释：唯一可能的三元组和为 0 。
 
提示：

3 <= nums.length <= 3000

-105 <= nums[i] <= 105

# 暴力解法：三重循环
不出意料的超时了！

In [ ]:
class Solution:
    def threeSum(self, nums: list[int]) -> list[list[int]]:
        ans = set()
        for i in range(len(nums)):
            for j in range(i+1, len(nums)):
                for k in range(j+1, len(nums)):
                    if nums[i] + nums[j] + nums[k] == 0:
                        ans.add(tuple(sorted([nums[i], nums[j], nums[k]])))
        return list(ans)

# 两数之和：先把两数之和用字典存起来，然后按照两数之和的思路写
还是超时了

In [ ]:
class Solution:
    def threeSum(self, nums: list[int]) -> list[list[int]]:
        ans = set()
        two_sum = collections.defaultdict(set)
        for i in range(len(nums)):
            for j in range(i+1, len(nums)):
                two_sum[nums[i] + nums[j]].add(tuple([i, j]))
        # print(two_sum)
        for i in range(len(nums)):
            two_nums = two_sum.get(-nums[i], None)
            if not two_nums:
                continue
            for two_num in two_nums:
                if i not in two_num:
                    print(nums[i], two_num)
                    ans.add(tuple(sorted([nums[i], nums[two_num[0]], nums[two_num[1]]])))
        return list(ans)
        

# 双指针
先从小到大排序，O(nlogn)
每次固定一个数，然后使用两个指针指向另外两个数，让左指针指向这个数的右边、右指针指向末尾。如果当前三个数的和：
- 大于0：说明大了，右指针往左移动一格
- 小于0：说明小了，左指针往右移动一格
- 等于0：用集合存储结果（去重）

In [ ]:
class Solution:
    def threeSum(self, nums: list[int]) -> list[list[int]]:
        ans = set()
        sorted_nums = sorted(nums)
        for i in range(len(sorted_nums)):
            left = i + 1
            right = len(sorted_nums) - 1
            while left < right:
                sum_3 = sorted_nums[i] + sorted_nums[left] + sorted_nums[right]
                if sum_3 > 0:
                    right -= 1
                elif sum_3 < 0:
                    left += 1
                elif sum_3 == 0:
                    ans.add(tuple([sorted_nums[i], sorted_nums[left], sorted_nums[right]]))
                    right -= 1
                    left += 1

        return list(ans)
        

## 优化
上面的代码已经能够成功通过了，但是观察数据模式，排序后数组是递增的，我们还可以在以下两个方面进行优化：
- 当第 i 个数大于0时，直接退出循环，因为后面的数只会比它大，再也凑不出相加为0的数了。
- 去重：排序之后所有相同的数都在一起了，所以可以对左右指针去重（有相等的数就一直循环迭代，左指针往右看，右指针往左看），此时得到的结果天然就是不同的，因此也不需要set了，而且时间复杂度也更低了。

In [ ]:
class Solution:
    def threeSum(self, nums: list[int]) -> list[list[int]]:
        nums.sort()  # 第一步：必须先排序
        ans = []
        n = len(nums)
        
        for i in range(n - 2):
            # 剪枝优化：如果当前遍历的数字已经大于 0，后面不可能加出 0 了
            if nums[i] > 0:
                break
                
            # 对 i 去重：如果和上一个数字相同，直接跳过，避免产生重复的结果
            if i > 0 and nums[i] == nums[i - 1]:
                continue
                
            # 初始化双指针
            left, right = i + 1, n - 1
            
            while left < right:
                s = nums[i] + nums[left] + nums[right]
                
                if s < 0:
                    left += 1  # 加起来太小了，左指针右移，让和变大
                elif s > 0:
                    right -= 1 # 加起来太大了，右指针左移，让和变小
                else:
                    # 找到了！
                    ans.append([nums[i], nums[left], nums[right]])
                    
                    # 对 left 和 right 去重（非常关键）
                    while left < right and nums[left] == nums[left + 1]:
                        left += 1
                    while left < right and nums[right] == nums[right - 1]:
                        right -= 1
                        
                    # 找到一组解后，双指针同时收缩
                    left += 1
                    right -= 1
                    
        return ans